# Dataset Inspection

## Load Datasets

In [3]:
from pathlib import Path
import cv2
import pandas as pd
from tqdm import tqdm

cv2.setNumThreads(0)

roots = [
    Path("~/ChipInspector/data/raw/real/DatasetA-Real").expanduser(),
    Path("~/ChipInspector/data/raw/real/DatasetB-Real").expanduser(),
]
classes = ["defective", "nondefective"]
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

records = []
missing_labels = []
corrupted_images = []

for root in roots:
    for cls in classes:
        if cls == "defective":
            img_dir = root / cls / "image"
            lbl_dir = root / cls / "label"
            label_map = {p.stem: p for p in lbl_dir.iterdir() if p.is_file()}
        else:
            # nondefective stores images directly with no image/label split
            img_dir = root / cls
            label_map = {}

        img_files = [p for p in img_dir.iterdir() if p.suffix.lower() in img_exts]

        for img_path in tqdm(img_files, desc=f"{root.name}/{cls}", mininterval=0.5):
            if cls == "defective":
                lbl_path = label_map.get(img_path.stem)
                if not lbl_path:
                    missing_labels.append(str(img_path))
            else:
                lbl_path = None

            img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
            if img is None:
                corrupted_images.append(str(img_path))
                continue

            h, w = img.shape[:2]
            channels = 1 if img.ndim == 2 else img.shape[2]

            records.append({
                "source": root.name,
                "class": cls,
                "image_path": str(img_path),
                "label_path": str(lbl_path) if lbl_path else None,
                "label_ext": lbl_path.suffix.lower() if lbl_path else None,
                "height": h,
                "width": w,
                "channels": channels,
                "dtype": str(img.dtype),
            })

df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nTotal combined valid images: {len(df)}")
print(f"Corrupted/unreadable images: {len(corrupted_images)}")
print(f"Defective images missing a label file: {len(missing_labels)}")

print("\n--- Breakdown by Source & Class ---")
print(df.groupby(["source", "class"]).size())

print("\n--- Label File Extensions (Defective) ---")
print(df["label_ext"].value_counts(dropna=False))

print("\n--- Channel Counts & Data Types ---")
print(df.value_counts(["channels", "dtype"]))

print("\n--- Top Image Resolutions (Height x Width) ---")
print(df.value_counts(["height", "width"]).head(5))


DatasetB-Real/nondefective: 100%|██████████| 909/909 [00:00<00:00, 1928.09it/s]


Total combined valid images: 3511
Corrupted/unreadable images: 0
Defective images missing a label file: 0

--- Breakdown by Source & Class ---
source         class       
DatasetA-Real  defective        780
               nondefective    1361
DatasetB-Real  defective        461
               nondefective     909
dtype: int64

--- Label File Extensions (Defective) ---
label_ext
None    2270
.xml    1241
Name: count, dtype: int64

--- Channel Counts & Data Types ---
channels  dtype
3         uint8    2141
1         uint8    1370
Name: count, dtype: int64

--- Top Image Resolutions (Height x Width) ---
height  width
270     258      1135
193     289      1091
379     378       746
276     264       235
281     373       126
Name: count, dtype: int64


In [4]:
df.head(5)

,source,class,image_path,label_path,label_ext,height,width,channels,dtype
0,DatasetA-Real,defective,/Users/haroonkhan/ChipInspector/data/raw/real/...,/Users/haroonkhan/ChipInspector/data/raw/real/...,.xml,379,378,3,uint8
1,DatasetA-Real,nondefective,/Users/haroonkhan/ChipInspector/data/raw/real/...,None,None,193,289,3,uint8
2,DatasetB-Real,defective,/Users/haroonkhan/ChipInspector/data/raw/real/...,/Users/haroonkhan/ChipInspector/data/raw/real/...,.xml,270,258,1,uint8
3,DatasetA-Real,nondefective,/Users/haroonkhan/ChipInspector/data/raw/real/...,None,None,193,289,3,uint8
4,DatasetB-Real,nondefective,/Users/haroonkhan/ChipInspector/data/raw/real/...,None,None,270,258,1,uint8
